<a href="https://colab.research.google.com/github/ojaswitasinha/price_reality_checker-ml/blob/main/price_reality_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Price Reality Checker


In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding="latin-1")

print("Loaded file:", filename)
print("Rows and columns:", df.shape)

Saving laptop_price.csv to laptop_price (1).csv
Loaded file: laptop_price (1).csv
Rows and columns: (1303, 13)


In [ ]:
display(df.head())

print("Shape(rows, columns):", df.shape)
print("Columns:", df.columns.tolist())

df.info()

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


Shape(rows, columns): (1303, 13)
Columns: ['laptop_ID', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price_euros']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Memory            1303 non-null   object 
 9   Gpu               1303 non-null   object 
 10  OpSys             1303 non-null   object 
 11  Weight            1303 non-null   object 
 12  Price_euros       1303 non-null   float64
dty

In [ ]:
target_col="Price_euros"

if target_col not in df.columns:
  raise ValueError(
      f"{target_col!r} is not in the dataset."
      f"Available columns: {df.columns.tolist()}"
  )

print("Price column:",target_col)
display(df[target_col].head())

Price column: Price_euros


,Price_euros
0,1339.69
1,898.94
2,575.00
3,2537.45
4,1803.60


Separate the price from the features

In [ ]:
import numpy as np
import pandas as pd

y_numeric= pd.to_numeric(df[target_col], errors="coerce")
valid_rows=y_numeric.notna()

print("Rows removed because price is missing or non-numeric:", int((~valid_rows).sum()))

data = df.loc[valid_rows].copy()
y = y_numeric.loc[valid_rows].astype(float)
X = data.drop(columns=[target_col])

print("Feature table shape:", X.shape)
print("Price summary:")
display(y.describe())

Rows removed because price is missing or non-numeric: 0
Feature table shape: (1303, 12)
Price summary:


,Price_euros
count,1303.000000
mean,1123.686992
std,699.009043
min,174.000000
25%,599.000000
50%,977.000000
75%,1487.880000
max,6099.000000


Split Training and test data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 1042
Test rows: 261


Set up preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features=make_column_selector(dtype_include=np.number)
categorical_features=make_column_selector(dtype_exclude=np.number)

numeric_preprocessor= Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_preprocessor= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor= ColumnTransformer(transformers=[("numeric", numeric_preprocessor, numeric_features),
                                              ("categorical", categorical_preprocessor, categorical_features)])

Create three models

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

baseline_model= Pipeline(steps=[("preprocessor", preprocessor),
                                ("model", DummyRegressor(strategy="mean"))])

linear_model= Pipeline(steps=[("preprocessor", preprocessor),
                              ("model", LinearRegression())])

random_forest_model= Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
          n_estimators=200,
          random_state=42,
          n_jobs=-1
    ))
])

models={
    "Median baseline": baseline_model,
    "Linear regression": linear_model,
    "Random forest": random_forest_model
}


Train and evaluate

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results=[]

for model_name, model in models.items():
   model.fit(X_train, y_train)
   predictions= model.predict(X_test)

   results.append({
       "Model": model_name,
       "MAE": mean_absolute_error(y_test, predictions),
       "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
       "R2": r2_score(y_test, predictions)
   })

results_df= pd.DataFrame(results).sort_values("MAE")
display(results_df)

,Model,MAE,RMSE,R2
2,Random forest,190.448972,360.936559,0.743516
1,Linear regression,213.979136,322.096391,0.795746
0,Median baseline,514.636391,713.068877,-0.001065


Compare actual and predicted prices

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

test_predictions = best_model.predict(X_test)

comparison = pd.DataFrame({
    "Actual price": y_test,
    "Predicted price": test_predictions,
    "Residual": y_test - test_predictions
})

print("Best model by MAE:", best_model_name)
display(comparison.head(10))

Best model by MAE: Random forest


,Actual price,Predicted price,Residual
479,1672.00,1070.68510,601.31490
1022,1149.00,1150.13510,-1.13510
298,499.00,604.42140,-105.42140
1265,899.00,848.69095,50.30905
774,1244.00,1282.76545,-38.76545
115,1398.99,1229.54000,169.45000
582,719.00,692.59665,26.40335
527,459.00,552.76375,-93.76375
380,1191.00,1236.93105,-45.93105
101,349.00,384.51330,-35.51330
